# Controlling MODFLOW 6 with the API — A. Basic usage

The MODFLOW 6 application programming interface (API) lets you drive a simulation from Python instead of running the `mf6` executable as a black box. Using the [`modflowapi`](https://github.com/MODFLOW-ORG/modflowapi) wrapper around the compiled MODFLOW 6 shared library, you can step through a model one time step (or even one solver iteration) at a time, read and modify internal variables while the model runs, and react to the simulated state.

This is the first notebook in a series on the API:

- **A. Basic usage** (this notebook) — run a model through the API, open the solver loop by hand, and drive a coupled flow-and-transport model.
- [**B. Monitoring convergence**](modflow-api-B.ipynb) — watch the solver converge live.
- [**C. Adjusting recharge with a callback**](modflow-api-C.ipynb) — change inputs on the fly.
- **D. Reverse drain** — a reverse-flow drain example (added separately).
- [**E. Streamflow augmentation**](modflow-api-E.ipynb) — implement a real-time operating rule.

Start here: this notebook establishes the API lifecycle and the mental model the others build on.

## How the API works: the big picture

Before diving into code, here is the mental model used throughout this notebook. Every example below is a variation on it, so it is worth a minute now.

**A MODFLOW 6 simulation is a nest of objects**

- a **simulation** contains one or more **models** — for example a groundwater-flow model (`GWF`) and a groundwater-transport model (`GWT`);
- each model contains **packages** — recharge (`RCH`), wells (`WELL`), streams (`SFR`), and so on;
- the models are advanced by one or more **solutions** (the numerical solvers), labelled `SLN_1`, `SLN_2`, ….

**Time is organized in three nested levels**

- a **stress period** is a block of time over which the boundary conditions (pumping, recharge, …) are held constant;
- each stress period is divided into one or more **time steps**;
- each time step is solved by repeating **outer (Picard) iterations** until the solution stops changing — i.e. it *converges*. The largest head change in an iteration is `HNCG`; the cap on the number of iterations is `MXITER`.

**Running a model through the API follows one lifecycle**

```text
mf6.initialize()                 # load the model into memory
while not finished:              # loop over time steps
    mf6.update()                 #   advance one time step (solves it internally)
mf6.finalize()                   # release memory and write the output files
```

`update()` hides the solver loop. When you need to reach *inside* a time step, you replace it with the manual loop:

```text
mf6.prepare_time_step(dt)
mf6.prepare_solve()
while not converged:             # outer iterations
    converged = mf6.solve()      #   one Picard iteration
mf6.finalize_solve()
mf6.finalize_time_step()
```

**Two ways to read and change data while the model runs**

- **High-level (callbacks):** `modflowapi` hands you the live simulation as familiar Python objects, so you can reach a package by name and edit its `stress_period_data` directly. This is the easiest way to work with the standard stress packages.
- **Low-level (direct memory access):** for variables the packages don't expose, you read MODFLOW's memory by *address* — a `(variable, model, package)` tag returned by `get_var_address`:
  - `get_value(tag)` returns a **copy**: a snapshot you can inspect without affecting the run, while
  - `get_value_ptr(tag)` returns a **pointer**: a live array that stays in sync with the running model and can be written back to change it.

Keep this picture — *simulation → models → packages*, the *initialize → update/solve → finalize* lifecycle, and *copy vs. pointer* — in mind as you go.

## Imports and setup

Import FloPy, the `modflowapi` interface, and the supporting plotting and data libraries, then create a directory for the figures saved by this notebook.

In [ ]:
%matplotlib inline
import os
import pathlib as pl
import platform

import flopy
from modflowapi import ModflowApi

In [ ]:
fig_path = pl.Path(".figures")
fig_path.mkdir(exist_ok=True)

In [ ]:
sample_frequency = "annual"  # monthly or annual
name = "sv"

### Locate the MODFLOW 6 library

The API drives MODFLOW 6 through its compiled shared library (`libmf6`), not the `mf6` command-line executable. Find the library (`.so` on Linux, `.dylib` on macOS, `.dll` on Windows) and the executable inside the active environment (here the project's pixi environment), and confirm that both exist.

In [ ]:
env_path = pl.Path(os.environ.get("CONDA_PREFIX", None))
assert env_path is not None, "Notebook must be run from the mf6xtd pixi environment"

bin_path = "bin"
exe_ext = ""
if "linux" in platform.platform().lower():
    lib_ext = ".so"
elif "darwin" in platform.platform().lower() or "macos" in platform.platform().lower():
    lib_ext = ".dylib"
else:
    bin_path = "Scripts"
    lib_ext = ".dll"
    exe_ext = ".exe"
lib_name = env_path / f"{bin_path}/libmf6{lib_ext}"
if not lib_name.is_file():
    lib_name = env_path / f"lib/libmf6{lib_ext}"
if not lib_name.is_file():
    raise FileNotFoundError(
        f"Could not find mf6 library at in either: {env_path / 'bin'}"
        + f" or {env_path / 'lib'}"
    )

mf6_exe = env_path / f"{bin_path}/mf6{exe_ext}"

In [ ]:
lib_name.is_file(), mf6_exe.is_file()

## Run a model with the API

The simplest use of the API: load an existing model, then drive it through time from Python. Instead of calling the `mf6` executable, initialize the library, repeatedly call `update()` to advance one time step until the end of the simulation is reached, and then finalize. This produces the same results as running the model with the executable.

In [ ]:
ws = pl.Path(
    f"../data/synthetic-valley/synthetic-valley-base-{sample_frequency}"
).resolve()
new_ws = pl.Path(f"models/synthetic-valley-base-{sample_frequency}_api01")

sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=ws, write_headers=False)
sim.set_sim_path(new_ws)
sim.write_simulation()

In [ ]:
mf6 = ModflowApi(lib_name, working_directory=new_ws)
mf6.initialize()

current_time = mf6.get_current_time()
end_time = mf6.get_end_time()

while current_time < end_time:
    print(f"Running simulation time: {current_time}")
    mf6.update()
    current_time = mf6.get_current_time()
mf6.finalize()

The loop above advanced the model one time step at a time and produced exactly the same results as running the `mf6` executable. Each `update()` call quietly ran the full solver loop for that time step — the next examples pry that loop open so you can act between iterations.

## Run a model with a manual solver loop

This example uses the watershed groundwater flow (GWF) model to show the manual solver loop. Instead of calling `update()`, it prepares each time step and then iterates the solver itself (`prepare_solve` / `solve` / `finalize_solve`) until the solution converges. Working at this level lets you read or modify the solution between outer iterations.

In [ ]:
ws = pl.Path("../data/watershed/gwf-only")
new_ws = pl.Path("models/watershed-gwf_api02")

sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=ws, write_headers=False)
sim.set_sim_path(new_ws)
sim.write_simulation()

In [ ]:
mf6 = ModflowApi(lib_name, working_directory=new_ws)
mf6.initialize()

# maximum outer iterations
mxit_tag = mf6.get_var_address("MXITER", "SLN_1")
max_iter = mf6.get_value(mxit_tag)

current_time = mf6.get_current_time()
end_time = mf6.get_end_time()

while current_time < end_time:
    print(f"Running simulation time: {current_time}")

    # get dt and prepare for non-linear iterations
    dt = mf6.get_time_step()
    mf6.prepare_time_step(dt)

    # convergence loop
    kiter = 0
    mf6.prepare_solve()

    while kiter < max_iter:
        # here is where you could update well rates or other time-varying
        # inputs that depend on the solution from the previous iteration.
        # For example, you could get the current head solution and use it to
        # update a well rate based on a specified head difference and conductance.

        # solve with updated well rate
        has_converged = mf6.solve()
        kiter += 1

        if has_converged:
            msg = f"  Component {1}" + f" converged in {kiter}" + " outer iterations"
            print(msg)
            break

    if not has_converged:
        print("model did not converge")

    # finalize time step
    mf6.finalize_solve()

    # finalize time step and update time
    mf6.finalize_time_step()
    current_time = mf6.get_current_time()

print("Finalize simulation.")
mf6.finalize()

## Run a coupled flow and transport model

Extend the manual solver loop to a simulation with more than one solution. The watershed model couples a groundwater flow (GWF) and a transport (GWT) model, so each time step loops over both solutions (`SLN_1` and `SLN_2`), solving each to convergence in turn.

In [ ]:
ws = pl.Path("../data/watershed/gwf-gwt")
new_ws = pl.Path("models/watershed-gwf-gwt_api03")

sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=ws, write_headers=False)
sim.set_sim_path(new_ws)
sim.write_simulation()

In [ ]:
mf6 = ModflowApi(lib_name, working_directory=new_ws)
mf6.initialize()

# maximum outer iterations
mxit_tag = mf6.get_var_address("MXITER", "SLN_1")
max_iter_gwf = mf6.get_value(mxit_tag)
mxit_tag = mf6.get_var_address("MXITER", "SLN_2")
max_iter_gwt = mf6.get_value(mxit_tag)


current_time = mf6.get_current_time()
end_time = mf6.get_end_time()

while current_time < end_time:
    print(f"Running simulation time: {current_time}")

    # get dt and prepare for non-linear iterations
    dt = mf6.get_time_step()
    mf6.prepare_time_step(dt)

    # convergence loop
    for km in (1, 2):  # loop over gwf and gwt
        kiter = 0
        max_iter = max_iter_gwf if km == 1 else max_iter_gwt
        mf6.prepare_solve(component_id=km)

        while kiter < max_iter:
            # as in the previous example, you could update inputs here between
            # outer iterations - separately for the flow (km == 1) and transport
            # (km == 2) solutions
            if km == 1:
                pass  # update gwf inputs if needed
            else:
                pass  # update gwt inputs if needed

            # solve with updated well rate
            has_converged = mf6.solve(component_id=km)
            kiter += 1

            if has_converged:
                msg = (
                    f"  Component {km}" + f" converged in {kiter}" + " outer iterations"
                )
                print(msg)
                break

        if not has_converged:
            print("model did not converge")

        # finalize time step
        mf6.finalize_solve(component_id=km)

    # finalize time step and update time
    mf6.finalize_time_step()
    current_time = mf6.get_current_time()

print("Finalize simulation.")
mf6.finalize()